# Notebook 04 — Training Pipeline / Model A

This is the canonical Notebook 04 entry point.

The validated training implementation is centralized in `src/training_pipeline.py` so the three production notebooks use **one** training algorithm rather than copy/pasted versions that can drift.

## Notebook 04 structure

- **This notebook**: frozen training-pipeline contract plus Model A production run (**D-081**)
- `04_training_pipeline_model_b.ipynb`: standalone Model B production run (**D-082**)
- `04_training_pipeline_model_c.ipynb`: standalone Model C production run (**D-083**)

Each notebook can start from a fresh Colab T4 runtime. It clones/pulls the repository, mounts the persistent Google Drive production directory, imports the same canonical training module, and then runs or verifies only its assigned model.

## Frozen Notebook 04 contract

The implementation in `src/training_pipeline.py` preserves the canonical Notebook 04 decisions **D-065 through D-080**:

- deterministic causal packing: 512-token inputs, one-token-shifted targets, stride 512
- 39,062 examples / 19,999,744 scored targets per epoch
- deterministic `seed + epoch` training shuffle
- final 22-sequence partial logical update is flushed, not dropped or duplicated
- effective batch: 16,384 targets/update = 32 × 512-token sequences
- AdamW: betas `(0.9, 0.95)`, epsilon `1e-8`, weight decay 0.10 for `ndim>=2`
- global gradient clipping at 1.0
- Tesla T4: FP16 autocast + GradScaler, FP32 master parameters
- peak learning rate `2e-3`, selected by the controlled Model A LR probe
- 5% warmup (183 updates), then cosine decay to 10% of peak
- 3 epochs / 3,663 optimizer updates / 59,999,232 target exposures per model
- validation every 200 updates plus epoch end
- official validation split only for tuning/checkpointing
- official test split remains untouched
- D-079 T4 preflight: physical micro-batch 32 for A, B, and C
- persistent best/latest checkpoints and run-history JSON artifacts

The detailed experimental record is `docs/decisions/04_training_pipeline.md`; the project-wide chronology is `docs/DECISION_INDEX.md`.

## Model A

Model A contains 7,407,872 parameters. If its verified persistent production artifacts already exist, this notebook will validate and reuse them instead of retraining.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

COLAB_ROOT = Path("/content")
REPO = COLAB_ROOT / "foundation-model-from-scratch"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/traderjohnd/foundation-model-from-scratch.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "origin", "main"],
        check=True,
    )

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

def ensure_package(import_name: str, install_spec: str):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", install_spec],
            check=True,
        )
        return importlib.import_module(import_name)

ensure_package("datasets", "datasets")
ensure_package("tokenizers", "tokenizers==0.23.1")

assert Path("src/model.py").exists()
assert Path("src/data.py").exists()
assert Path("src/training_pipeline.py").exists()
assert Path("results/tokenizer/tokenizer.json").exists()

print("Repository/bootstrap audit: PASS")
print("Working directory:", os.getcwd())


In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")

try:
    if not (DRIVE_ROOT / "MyDrive").exists():
        drive.mount(str(DRIVE_ROOT), force_remount=False)
except Exception:
    drive.mount(str(DRIVE_ROOT), force_remount=True)

PERSISTENT_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
    / "foundation-model-from-scratch"
    / "production"
)
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

print("Persistent production root:")
print(PERSISTENT_ROOT)


In [ ]:
from src.training_pipeline import (
    audit_persistent_model_artifacts,
    run_production_model,
)

model_a_summary = run_production_model(
    "A",
    PERSISTENT_ROOT,
    require_t4=True,
)

audit_a = audit_persistent_model_artifacts(
    "A",
    PERSISTENT_ROOT,
)

assert model_a_summary["completed_updates"] == 3_663
assert model_a_summary["completed_full_epochs"] == 3
assert model_a_summary["total_target_exposures"] == 59_999_232
assert model_a_summary["validation_events"] == 21
assert model_a_summary["official_test_split_content_used"] is False

print()
print("MODEL A PERSISTENT ARTIFACT AUDIT: PASS")
print(
    "Best validation loss:",
    f'{model_a_summary["best_validation_loss"]:.6f}',
)
print(
    "Best perplexity:",
    f'{model_a_summary["best_validation_perplexity"]:.2f}',
)
print(
    "Best update:",
    model_a_summary["best_validation_update"],
)
print("Official test split used: NO")
